In [ ]:
# =============
# LIBRARY IMPORTS

from astropy.io import fits
import glob
import numpy as np
import os
import pickle
import random
import sys

In [ ]:
# =============
# SPECTRA INTERPOLATION AND PICKLE SERIALIZATION

folder_path = r'spectrums'
batch_size = 10000  # Numero de archivos a procesar por lote
index_file = "extra/batch_index.txt"

# Verify that batch_index.txt exists
if not os.path.exists(index_file):
    print(f"Error: File {index_file} does not exist. Halting execution.")
    sys.exit(1)

# Attempt to read and parse content as integer
with open(index_file, "r") as f:
    content = f.read().strip()
    try:
        batch_index = int(content)
    except ValueError:
        print(f"Error: File {index_file} does not contain a valid integer. Halting execution.")
        sys.exit(1)

def expand_points(wavelength, flux, target_count=5000):
    # Convert to lists for insertion
    wl = list(wavelength)
    fl = list(flux)
    
    # Compute absolute differences between consecutive points
    diffs = [abs(fl[i+1] - fl[i]) for i in range(len(fl)-1)]
    # Get indices sorted in descending order of difference
    sorted_indices = sorted(range(len(diffs)), key=lambda i: diffs[i], reverse=True)
    
    # Insert new points using sorted indices
    while len(wl) < target_count:
        # Traverse indices in descending order to avoid shift issues
        for idx in sorted_indices:
            if len(wl) >= target_count:
                break
            # Compute linear interpolation between index idx and idx+1
            new_wl = (wl[idx] + wl[idx+1]) / 2
            new_fl = (fl[idx] + fl[idx+1]) / 2
            # Insert new point at corresponding index
            wl.insert(idx+1, new_wl)
            fl.insert(idx+1, new_fl)
    
    return np.array(wl), np.array(fl)

# Get initial list of FITS files
files = [f for f in os.listdir(folder_path) if f.endswith('.fits')]

while files:
    # Randomly sample current batch (take all if fewer than batch_size remain)
    if len(files) < batch_size:
        current_batch = files.copy()
    else:
        current_batch = random.sample(files, batch_size)
    
    # Storage dictionary for current batch
    spectra_data = {}
    i = 0

    for filename in current_batch:
        file_path = os.path.join(folder_path, filename)
        try:
            with fits.open(file_path) as hdul:
                # Verify file contains expected HDU extensions
                if len(hdul) > 2:
                    flux_data = hdul[1].data["flux"]            # Flux data
                    loglam_data = hdul[1].data["loglam"]        # log(lambda) data
                    wavelength_data = 10 ** loglam_data         # Convert log(lambda) to wavelength
                    redshift = hdul[2].data["Z"][0]             # Extract redshift
                    
                    # Apply interpolation to reach 5000 points
                    interp_wavelength, interp_flux = expand_points(wavelength_data, flux_data, target_count=5000)
                    
                    # Save data
                    spectra_data[filename] = {
                        "wavelength": interp_wavelength,
                        "flux": interp_flux,
                        "redshift": redshift
                    }
                    
                    i += 1
                    if i % 1000 == 0:
                        print(f"Processed {filename} ({i})")
        except Exception as e:
            print(f"Error processing {filename}: {e}")
    
    print(f"Processed {len(spectra_data)} FITS files in batch {batch_index}.")
    
    # Save dictionary to pickle file
    output_file = f'data/spectra_data_complete{batch_index}.pkl'
    with open(output_file, 'wb') as f:
        pickle.dump(spectra_data, f)
    print(f"Data saved to {output_file}")
    
    # Save list of processed files for this batch
    batch_list_file = f'extra/batch_files_list{batch_index}.pkl'
    with open(batch_list_file, 'wb') as f:
        pickle.dump(current_batch, f)
    print(f"Batch {batch_index} file list saved to {batch_list_file}")
    
    # Remove newly processed files from file list
    files = [f for f in files if f not in current_batch]
    
    # Update batch_index.txt for next iteration
    batch_index += 1
    with open(index_file, "w") as f:
        f.write(str(batch_index))
    
print("All files in folder have been processed.")

In [ ]:
# =============
# REMOVAL OF PROCESSED .fits FILES

pickle_folder = 'extra'
spectrums_folder = r'spectrums'

# Get list of pickle files matching 'batch_files_list*.pkl'
pickle_files = [f for f in os.listdir(pickle_folder) if f.startswith('batch_files_list') and f.endswith('.pkl')]

# Iterate through each pickle file
for pickle_file in pickle_files:
    pickle_path = os.path.join(pickle_folder, pickle_file)
    print(f"Processing {pickle_path}...")
    try:
        with open(pickle_path, 'rb') as f:
            file_list = pickle.load(f)
    except Exception as e:
        print(f"Error opening {pickle_path}: {e}")
        continue
    # Iterate through names and delete corresponding files in spectrums folder
    for filename in file_list:
        file_path = os.path.join(spectrums_folder, filename)
        if os.path.exists(file_path):
            try:
                os.remove(file_path)
                print(f"Deleted: {file_path}")
            except Exception as e:
                print(f"Error deleting {file_path}: {e}")
        else:
            print(f"File not found: {file_path}")

print("Deletion process complete.")

In [ ]:
# =============
# CONVERSION A NUMPY ARRAYS

# Search for all files matching prefix in 'data' directory
data_files = glob.glob(os.path.join('data', 'spectra_data_complete*.pkl'))
data_files = sorted(data_files)
print(f"Found {len(data_files)} pickle files.")

# Count total number of spectra
total_spectra = 0
for file in data_files:
    with open(file, 'rb') as f:
        data_part = pickle.load(f)
    total_spectra += len(data_part)
print(f"Total spectra: {total_spectra}")

# Preallocate memmap files for flux, wavelength, and redshift in 'data' directory
flux_mmap_path = os.path.join('data', 'spectra_data_complete_flux.dat')
wavelength_mmap_path = os.path.join('data', 'spectra_data_complete_wavelength.dat')
redshift_mmap_path = os.path.join('data', 'spectra_data_complete_redshift.dat')

num_points = 5000
flux_mmap = np.memmap(flux_mmap_path, dtype='float32', mode='w+', shape=(total_spectra, num_points))
wavelength_mmap = np.memmap(wavelength_mmap_path, dtype='float32', mode='w+', shape=(total_spectra, num_points))
redshift_mmap = np.memmap(redshift_mmap_path, dtype='float32', mode='w+', shape=(total_spectra,))

# Load data from pickle files and write into memmaps
current_index = 0
for file in data_files:
    with open(file, 'rb') as f:
        data_part = pickle.load(f)
    print(f"Processing {file} with {len(data_part)} spectra...")
    for key in data_part:
        spectrum = data_part[key]
        flux = spectrum["flux"]              # 5000-point array
        wavelength = spectrum["wavelength"]  # 5000-point array
        redshift = spectrum["redshift"]      # Scalar value
        
        # Store into memmap
        flux_mmap[current_index, :] = flux
        wavelength_mmap[current_index, :] = wavelength
        redshift_mmap[current_index] = redshift
        
        current_index += 1

# Flush all changes to disk
flux_mmap.flush()
wavelength_mmap.flush()
redshift_mmap.flush()

print("Data successfully saved to memmap files inside 'data' folder.")

In [ ]:
# =============
# SPECTRA INTERPOLATION AND PICKLE SERIALIZATION (NED VARIANT)

folder_path = r'spectrums NASA'
batch_size = 100000
index_file = "extra/batch_index.txt"

# Verify existence of batch_index.txt
if not os.path.exists(index_file):
    print(f"Error: File {index_file} does not exist. Halting execution.")
    sys.exit(1)

# Read and validate batch_index
with open(index_file, "r") as f:
    content = f.read().strip()
    try:
        batch_index = int(content)
    except ValueError:
        print(f"Error: File {index_file} does not contain a valid integer. Halting execution.")
        sys.exit(1)

def expand_points(wavelength, flux, target_count=5000):
    wl = list(wavelength)
    fl = list(flux)
    # Absolute differences between consecutive points
    diffs = [abs(fl[i+1] - fl[i]) for i in range(len(fl)-1)]
    sorted_indices = sorted(range(len(diffs)), key=lambda i: diffs[i], reverse=True)
    # Insert until target_count is reached
    while len(wl) < target_count:
        for idx in sorted_indices:
            if len(wl) >= target_count:
                break
            new_wl = (wl[idx] + wl[idx+1]) / 2
            new_fl = (fl[idx] + fl[idx+1]) / 2
            wl.insert(idx+1, new_wl)
            fl.insert(idx+1, new_fl)
    return np.array(wl), np.array(fl)

# List of FITS files
files = [f for f in os.listdir(folder_path) if f.endswith('.fits')]

while files:
    current_batch = files.copy() if len(files) < batch_size else random.sample(files, batch_size)
    spectra_data = {}
    processed = 0

    for filename in current_batch:
        file_path = os.path.join(folder_path, filename)
        try:
            with fits.open(file_path) as hdul:
                try:
                    data    = hdul[0].data      # shape (5,3856)
                    header0 = hdul[0].header

                    # Construct log-linear wavelength vector
                    n_pix  = header0['NAXIS1']
                    coeff0 = header0['COEFF0']
                    coeff1 = header0['COEFF1']
                    crpix1 = header0.get('CRPIX1', 1)
                    pix    = np.arange(n_pix)
                    wave   = 10**(coeff0 + coeff1 * (pix + 1 - crpix1))
                    flux  = data[1, :]
                    redshift   = header0.get('Z')

                    # Clean NaNs before expansion
                    mask = np.isfinite(flux)
                    wavelength_data = wave[mask]
                    flux_data = flux[mask]

                except Exception as e:
                    continue

                # Interpolate up to 5000 points in flux
                interp_wavelength = wavelength_data
                interp_flux = flux_data
                # Apply expand_points in loop
                while interp_flux.shape[0] < 5000:
                    interp_wavelength, interp_flux = expand_points(interp_wavelength, interp_flux, target_count=5000)

                spectra_data[filename] = {
                    "wavelength": interp_wavelength,
                    "flux": interp_flux,
                    "redshift": redshift
                }
                processed += 1
                if processed % 1000 == 0:
                    print(f"Processed {processed} spectra in this batch.")
        except Exception as e:
            print(f"Error procesando {filename}: {e}")  # proceed to next file

    print(f"Processed {len(spectra_data)} FITS files in batch {batch_index}.")

    # Save results
    output_file = f'data/spectra_data_NASA{batch_index}.pkl'
    with open(output_file, 'wb') as f:
        pickle.dump(spectra_data, f)
    print(f"Data saved to {output_file}")

    batch_list_file = f'extra/batch_files_list{batch_index}.pkl'
    with open(batch_list_file, 'wb') as f:
        pickle.dump(current_batch, f)
    print(f"Batch {batch_index} file list saved to {batch_list_file}")

    # Update list and batch_index
    files = [f for f in files if f not in current_batch]
    batch_index += 1
    with open(index_file, "w") as f:
        f.write(str(batch_index))

print("All files in folder have been processed.")